## 1. Import Libraries

In [4]:
# Setup Python path to import sgg_benchmark
import sys
import os

# Get the absolute path to the sgg root directory
current_file = os.path.abspath(os.getcwd())
sgg_root = os.path.dirname(current_file)  # Go up from demo to sgg

# Add sgg root to Python path
if sgg_root not in sys.path:
    sys.path.insert(0, sgg_root)

# Change working directory to sgg root for relative imports
os.chdir(sgg_root)

print(f"Added to Python path: {sgg_root}")
print(f"Changed working directory to: {os.getcwd()}")
print(f"sgg_benchmark exists: {os.path.exists(os.path.join(sgg_root, 'sgg_benchmark'))}")

Added to Python path: /mnt/f/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/my_script
Changed working directory to: /mnt/f/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/my_script
sgg_benchmark exists: False


In [1]:
import os
import yaml
import torch
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO
from datetime import datetime
from PIL import Image
import cv2

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print(f"✓ Libraries imported successfully")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

✓ Libraries imported successfully
✓ PyTorch version: 2.5.1+cu124
✓ CUDA available: True
✓ GPU: NVIDIA GeForce GTX 1650


## 2. Configuration

In [6]:
# Paths
MODEL_PATH = '../checkpoints/my_best.pt'
DATA_CONFIG = '../datasets/psg/YOLO_anno/data.yaml'

# Results directory
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RESULTS_DIR = Path(f'test_results_{TIMESTAMP}')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Model path: {MODEL_PATH}")
print(f"📁 Data config: {DATA_CONFIG}")
print(f"📁 Results will be saved to: {RESULTS_DIR}")

📁 Model path: ../checkpoints/my_best.pt
📁 Data config: ../datasets/psg/YOLO_anno/data.yaml
📁 Results will be saved to: test_results_20251210_193506


## 3. Load Model & Dataset Info

In [7]:
# Load model
print("🔄 Loading model...")
model = YOLO(MODEL_PATH)
print("✓ Model loaded successfully")

# Load data config
with open(DATA_CONFIG, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"\n📊 Dataset Information:")
print(f"  Number of classes: {data_config['nc']}")
print(f"  Train set: {data_config['train']}")
print(f"  Val set: {data_config['val']}")
print(f"  Test set: {data_config['test']}")

# Display class names
class_names = data_config['names']
print(f"\n📋 Classes ({len(class_names)}):")
for i, name in enumerate(class_names[:10]):
    print(f"  {i}: {name}")
if len(class_names) > 10:
    print(f"  ... and {len(class_names) - 10} more classes")

🔄 Loading model...
✓ Model loaded successfully

📊 Dataset Information:
  Number of classes: 133
  Train set: /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/train/images
  Val set: /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/val/images
  Test set: /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/test/images

📋 Classes (133):
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  ... and 123 more classes


## 4. Run Validation on Test Set

In [9]:
print("🔍 Running validation on test set...\n")

# Run validation
test_results = model.val(
    data=DATA_CONFIG,
    split='test',
    batch=16,
    imgsz=640,
    conf=0.001,  # Low confidence threshold for evaluation
    iou=0.6,     # IoU threshold for NMS
    plots=True,
    save_json=True,
    verbose=True
)

print("\n✓ Validation completed!")

🔍 Running validation on test set...

Ultralytics 8.3.100 🚀 Python-3.11.14 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)


FileNotFoundError: 
Dataset '../datasets/psg/YOLO_anno/data.yaml' images not found ⚠️, missing path '/mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/val/images'
Note dataset download directory is '/mnt/f/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets'. You can update this in '/home/hieultph/.config/Ultralytics/settings.json'

## 5. Extract & Display Metrics

In [ ]:
# Extract metrics
results_dict = test_results.results_dict

# Main metrics
metrics = {
    'Precision': results_dict.get('metrics/precision(B)', 0),
    'Recall': results_dict.get('metrics/recall(B)', 0),
    'mAP@0.5': results_dict.get('metrics/mAP50(B)', 0),
    'mAP@0.5:0.95': results_dict.get('metrics/mAP50-95(B)', 0),
}

# Display metrics
print("\n" + "="*60)
print("📈 TEST SET EVALUATION RESULTS")
print("="*60)

for metric_name, value in metrics.items():
    print(f"{metric_name:20s}: {value:.4f} ({value*100:.2f}%)")

print("="*60)

# Save metrics to JSON
metrics_path = RESULTS_DIR / 'test_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"\n✓ Metrics saved to: {metrics_path}")

## 6. Per-Class Performance Analysis

In [ ]:
# Get per-class metrics
if hasattr(test_results, 'box'):
    box_metrics = test_results.box
    
    # Create DataFrame for per-class metrics
    per_class_data = {
        'Class': class_names,
        'Precision': box_metrics.p if hasattr(box_metrics, 'p') else [0] * len(class_names),
        'Recall': box_metrics.r if hasattr(box_metrics, 'r') else [0] * len(class_names),
        'mAP@0.5': box_metrics.ap50 if hasattr(box_metrics, 'ap50') else [0] * len(class_names),
        'mAP@0.5:0.95': box_metrics.ap if hasattr(box_metrics, 'ap') else [0] * len(class_names),
    }
    
    df_per_class = pd.DataFrame(per_class_data)
    
    # Sort by mAP@0.5
    df_per_class = df_per_class.sort_values('mAP@0.5', ascending=False)
    
    print("\n📊 Per-Class Performance (Top 20):")
    print(df_per_class.head(20).to_string(index=False))
    
    # Save to CSV
    csv_path = RESULTS_DIR / 'per_class_metrics.csv'
    df_per_class.to_csv(csv_path, index=False)
    print(f"\n✓ Per-class metrics saved to: {csv_path}")
else:
    print("⚠ Per-class metrics not available")

## 7. Visualize Metrics

In [ ]:
# Plot 1: Overall Metrics Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))

metric_names = list(metrics.keys())
metric_values = [metrics[k] for k in metric_names]

bars = ax.bar(metric_names, metric_values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Overall Test Set Performance', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'overall_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Plot saved to: {RESULTS_DIR / 'overall_metrics.png'}")

In [ ]:
# Plot 2: Top 15 & Bottom 15 Classes by mAP@0.5
if 'df_per_class' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Top 15 classes
    top_15 = df_per_class.head(15)
    axes[0].barh(range(len(top_15)), top_15['mAP@0.5'], color='green', alpha=0.7)
    axes[0].set_yticks(range(len(top_15)))
    axes[0].set_yticklabels(top_15['Class'], fontsize=9)
    axes[0].set_xlabel('mAP@0.5', fontsize=11)
    axes[0].set_title('Top 15 Classes by mAP@0.5', fontsize=12, fontweight='bold')
    axes[0].invert_yaxis()
    axes[0].grid(axis='x', alpha=0.3)
    
    # Bottom 15 classes
    bottom_15 = df_per_class.tail(15)
    axes[1].barh(range(len(bottom_15)), bottom_15['mAP@0.5'], color='red', alpha=0.7)
    axes[1].set_yticks(range(len(bottom_15)))
    axes[1].set_yticklabels(bottom_15['Class'], fontsize=9)
    axes[1].set_xlabel('mAP@0.5', fontsize=11)
    axes[1].set_title('Bottom 15 Classes by mAP@0.5', fontsize=12, fontweight='bold')
    axes[1].invert_yaxis()
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'top_bottom_classes.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Plot saved to: {RESULTS_DIR / 'top_bottom_classes.png'}")

In [ ]:
# Plot 3: Precision-Recall Scatter
if 'df_per_class' in locals():
    fig, ax = plt.subplots(figsize=(10, 8))
    
    scatter = ax.scatter(df_per_class['Recall'], 
                        df_per_class['Precision'],
                        c=df_per_class['mAP@0.5'],
                        s=100,
                        cmap='viridis',
                        alpha=0.6,
                        edgecolors='black',
                        linewidth=0.5)
    
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Distribution (colored by mAP@0.5)', 
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('mAP@0.5', fontsize=11)
    
    # Add diagonal line (F1 reference)
    ax.plot([0, 1], [0, 1], 'r--', alpha=0.3, linewidth=1, label='Equal P/R')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'precision_recall_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Plot saved to: {RESULTS_DIR / 'precision_recall_scatter.png'}")

## 8. Statistical Summary

In [ ]:
if 'df_per_class' in locals():
    print("\n📊 Statistical Summary of Per-Class Performance:\n")
    
    stats = df_per_class[['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']].describe()
    print(stats.to_string())
    
    # Additional statistics
    print("\n" + "="*60)
    print("Additional Statistics:")
    print("="*60)
    
    for col in ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']:
        median = df_per_class[col].median()
        q1 = df_per_class[col].quantile(0.25)
        q3 = df_per_class[col].quantile(0.75)
        print(f"\n{col}:")
        print(f"  Median: {median:.4f}")
        print(f"  Q1 (25%): {q1:.4f}")
        print(f"  Q3 (75%): {q3:.4f}")
        print(f"  IQR: {q3-q1:.4f}")
    
    # Count classes with good/poor performance
    print("\n" + "="*60)
    print("Performance Distribution (based on mAP@0.5):")
    print("="*60)
    excellent = (df_per_class['mAP@0.5'] >= 0.75).sum()
    good = ((df_per_class['mAP@0.5'] >= 0.5) & (df_per_class['mAP@0.5'] < 0.75)).sum()
    fair = ((df_per_class['mAP@0.5'] >= 0.25) & (df_per_class['mAP@0.5'] < 0.5)).sum()
    poor = (df_per_class['mAP@0.5'] < 0.25).sum()
    
    print(f"  Excellent (≥0.75): {excellent} classes ({excellent/len(df_per_class)*100:.1f}%)")
    print(f"  Good (0.5-0.75): {good} classes ({good/len(df_per_class)*100:.1f}%)")
    print(f"  Fair (0.25-0.5): {fair} classes ({fair/len(df_per_class)*100:.1f}%)")
    print(f"  Poor (<0.25): {poor} classes ({poor/len(df_per_class)*100:.1f}%)")

## 9. Test on Sample Images (Optional)

In [ ]:
# Uncomment and run this cell to test on specific images

# TEST_IMAGES_DIR = '../datasets/psg/test/images'  # Change to your test images directory
# test_images = list(Path(TEST_IMAGES_DIR).glob('*.jpg'))[:5]  # Get first 5 images

# if test_images:
#     print(f"🖼️ Testing on {len(test_images)} sample images...\n")
    
#     # Run inference
#     results = model.predict(
#         source=test_images,
#         conf=0.25,
#         iou=0.6,
#         save=True,
#         project=str(RESULTS_DIR),
#         name='sample_predictions',
#         verbose=False
#     )
    
#     # Display results
#     for i, (img_path, result) in enumerate(zip(test_images, results)):
#         print(f"\nImage {i+1}: {img_path.name}")
#         print(f"  Detected {len(result.boxes)} objects")
        
#         # Count detections per class
#         if len(result.boxes) > 0:
#             classes = result.boxes.cls.cpu().numpy()
#             unique, counts = np.unique(classes, return_counts=True)
#             for cls_id, count in zip(unique, counts):
#                 print(f"    {class_names[int(cls_id)]}: {count}")
    
#     print(f"\n✓ Prediction results saved to: {RESULTS_DIR / 'sample_predictions'}")
# else:
#     print("⚠ No test images found. Update TEST_IMAGES_DIR path.")

## 10. Generate Final Report

In [ ]:
# Generate comprehensive report
report = {
    'model_path': MODEL_PATH,
    'test_date': datetime.now().isoformat(),
    'dataset': {
        'num_classes': data_config['nc'],
        'test_set': data_config['test']
    },
    'overall_metrics': metrics,
}

if 'df_per_class' in locals():
    report['per_class_statistics'] = {
        'mean_precision': float(df_per_class['Precision'].mean()),
        'mean_recall': float(df_per_class['Recall'].mean()),
        'mean_mAP50': float(df_per_class['mAP@0.5'].mean()),
        'mean_mAP50_95': float(df_per_class['mAP@0.5:0.95'].mean()),
        'median_mAP50': float(df_per_class['mAP@0.5'].median()),
        'excellent_classes': int(excellent),
        'good_classes': int(good),
        'fair_classes': int(fair),
        'poor_classes': int(poor),
    }
    
    # Best and worst performing classes
    report['best_classes'] = df_per_class.head(10)[['Class', 'mAP@0.5']].to_dict('records')
    report['worst_classes'] = df_per_class.tail(10)[['Class', 'mAP@0.5']].to_dict('records')

# Save report
report_path = RESULTS_DIR / 'evaluation_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print("\n" + "="*60)
print("✓ EVALUATION COMPLETED")
print("="*60)
print(f"\n📁 All results saved to: {RESULTS_DIR}")
print(f"\n📄 Files generated:")
for file in sorted(RESULTS_DIR.glob('*')):
    print(f"  - {file.name}")
print("\n✓ Final report saved to:", report_path)

## Summary

Notebook này đã thực hiện:

1. ✅ Load mô hình YOLO từ checkpoint
2. ✅ Chạy validation trên test set
3. ✅ Tính toán các metrics: Precision, Recall, mAP@0.5, mAP@0.5:0.95
4. ✅ Phân tích performance theo từng class
5. ✅ Visualize kết quả bằng các biểu đồ
6. ✅ Tạo báo cáo chi tiết và lưu kết quả

Tất cả kết quả được lưu trong thư mục `test_results_[timestamp]/`